<div style="background-color:#000047; padding:30px; border-radius:10px; color:white; text-align:center;">
    <img src='Figures/alinco_white_text.png' style="height:100px; margin-bottom:10px;"/>
    <h1>Módulo 3: Modelos de Lenguaje</h1>
    <h2> Gradio: Construcción y Despliegue de Interfaces para Modelos de ML/NLP</h2>
</div>

**Gradio** es una librería de Python de código abierto que permite crear, en pocas líneas de código, interfaces web interactivas para modelos de Machine Learning, modelos de lenguaje (NLP), APIs o cualquier función de Python.

Es ampliamente utilizada para:
- **Demostrar** modelos rápidamente sin necesidad de saber HTML/CSS/JavaScript.
- **Compartir** demos con un enlace público temporal.
- **Desplegar** aplicaciones permanentes en *Hugging Face Spaces*.

---

## Documentación oficial y recursos

| Recurso | Enlace |
|---|---|
| Documentación oficial | https://www.gradio.app/docs |
| Guías (Getting Started) | https://www.gradio.app/guides/quickstart |
| Despliegue en Hugging Face Spaces | https://huggingface.co/docs/hub/spaces-sdks-gradio |
| Galería de ejemplos | https://www.gradio.app/demos |
| Repositorio de GitHub | https://github.com/gradio-app/gradio |
| Componentes disponibles | https://www.gradio.app/docs/gradio/introduction |

## 1. Instalación

Ejecuta la siguiente celda para instalar Gradio. Si ya lo tienes instalado, puedes omitirla.

In [ ]:
# Instalación de Gradio
%pip install --quiet gradio

import gradio as gr
print("Versión de Gradio:", gr.__version__)

## 2. Conceptos fundamentales

Gradio ofrece dos formas principales de construir una aplicación:

### `gr.Interface` (alto nivel)
La forma más rápida. Envuelve una **función de Python** y le asigna automáticamente componentes de entrada (`inputs`) y salida (`outputs`).

```python
gr.Interface(fn=mi_funcion, inputs="text", outputs="text")
```

### `gr.Blocks` (bajo nivel)
Mayor control sobre el diseño (filas, columnas, pestañas, eventos personalizados). Ideal para aplicaciones más complejas.

### Componentes principales
| Componente | Uso típico |
|---|---|
| `gr.Textbox` | Entrada/salida de texto |
| `gr.Number` / `gr.Slider` | Valores numéricos |
| `gr.Image` | Imágenes |
| `gr.Audio` | Audio |
| `gr.File` | Archivos |
| `gr.Dropdown` / `gr.Radio` / `gr.Checkbox` | Selección de opciones |
| `gr.Label` / `gr.JSON` | Salidas de clasificación o estructuradas |
| `gr.Chatbot` | Interfaces conversacionales |

## 3. Ejemplo 1: "Hola Mundo" con `gr.Interface`

Una función simple que recibe un nombre y devuelve un saludo.

> **Nota sobre la ejecución:** Al lanzar la app con `demo.launch()` se inicia un servidor local. En un notebook, la interfaz se mostrará embebida debajo de la celda. Usa `demo.close()` para detener el servidor cuando termines.

In [ ]:
import gradio as gr

def saludar(nombre):
    return f"¡Hola, {nombre}! Bienvenido a Gradio."

demo = gr.Interface(
    fn=saludar,
    inputs=gr.Textbox(label="Tu nombre", placeholder="Escribe aquí..."),
    outputs=gr.Textbox(label="Saludo"),
    title="Saludo con Gradio",
    description="Escribe tu nombre y recibe un saludo."
)

demo.launch()

## 4. Ejemplo 2: Aplicación de NLP — Análisis de Sentimiento

Usaremos un *pipeline* de Hugging Face Transformers para clasificar el sentimiento de un texto. Esto muestra el caso de uso más común en NLP: **envolver un modelo en una interfaz**.

Si no tienes `transformers` instalado, ejecuta la primera línea.

In [ ]:
%pip install --quiet transformers torch

import gradio as gr
from transformers import pipeline

# Modelo multilingüe de análisis de sentimiento
clasificador = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

def analizar_sentimiento(texto):
    if not texto.strip():
        return {}
    resultado = clasificador(texto)[0]
    # Devolvemos un diccionario {etiqueta: probabilidad} para usar gr.Label
    return {resultado["label"]: float(resultado["score"])}

demo_sentimiento = gr.Interface(
    fn=analizar_sentimiento,
    inputs=gr.Textbox(lines=4, label="Texto a analizar",
                      placeholder="Escribe una reseña o comentario..."),
    outputs=gr.Label(label="Sentimiento (1 a 5 estrellas)"),
    title="Análisis de Sentimiento Multilingüe",
    description="Clasifica el texto en una escala de 1 a 5 estrellas.",
    examples=[
        ["Me encantó este producto, lo recomiendo totalmente."],
        ["Es terrible, una pérdida de dinero."],
        ["Está bien, cumple su función."]
    ]
)

demo_sentimiento.launch()

## 5. Ejemplo 3: Diseño personalizado con `gr.Blocks`

Con `gr.Blocks` controlamos el diseño usando filas (`gr.Row`) y columnas (`gr.Column`), y conectamos eventos manualmente con `.click()`.

En este ejemplo creamos una pequeña herramienta de procesamiento de texto: contar palabras, contar caracteres y convertir a mayúsculas.

In [ ]:
import gradio as gr

def procesar(texto):
    num_palabras = len(texto.split())
    num_caracteres = len(texto)
    mayusculas = texto.upper()
    return num_palabras, num_caracteres, mayusculas

with gr.Blocks(title="Herramienta de Texto") as demo_blocks:
    gr.Markdown("# 🛠️ Herramienta de Procesamiento de Texto")
    gr.Markdown("Escribe un texto y obtén estadísticas básicas.")

    with gr.Row():
        with gr.Column():
            entrada = gr.Textbox(lines=5, label="Texto de entrada")
            boton = gr.Button("Procesar", variant="primary")
        with gr.Column():
            out_palabras = gr.Number(label="Número de palabras")
            out_caracteres = gr.Number(label="Número de caracteres")
            out_mayus = gr.Textbox(label="En mayúsculas")

    boton.click(
        fn=procesar,
        inputs=entrada,
        outputs=[out_palabras, out_caracteres, out_mayus]
    )

demo_blocks.launch()

## 6. Ejemplo 4: Chatbot con `gr.ChatInterface`

Gradio incluye `gr.ChatInterface`, una abstracción específica para crear interfaces de chat (muy útil con LLMs). La función debe recibir `(message, history)` y devolver la respuesta del asistente.

Aquí usamos una respuesta simulada (eco) para no depender de una API externa. En un caso real reemplazarías la lógica por una llamada a un modelo (OpenAI, Hugging Face, etc.).

In [ ]:
import gradio as gr

def responder(message, history):
    # history es una lista de mensajes previos (formato 'messages')
    # Aquí simulamos una respuesta sencilla.
    return f"Recibí tu mensaje: '{message}'. (Aquí iría la respuesta del modelo)"

demo_chat = gr.ChatInterface(
    fn=responder,
    type="messages",
    title="Chatbot de demostración",
    description="Reemplaza la función 'responder' por una llamada a tu LLM."
)

demo_chat.launch()

## 7. Despliegue

### 7.1 Enlace público temporal (`share=True`)
La forma más rápida de compartir tu demo. Gradio genera un enlace público (`*.gradio.live`) válido por **72 horas**. Útil para mostrar resultados rápidamente, pero **no** para producción.

```python
demo.launch(share=True)
```

### 7.2 Parámetros útiles de `launch()`
| Parámetro | Descripción |
|---|---|
| `share=True` | Genera enlace público temporal |
| `server_name="0.0.0.0"` | Expone la app en la red local |
| `server_port=7860` | Define el puerto |
| `auth=("usuario", "clave")` | Protege con usuario y contraseña |
| `debug=True` | Muestra errores detallados (útil en notebooks) |
| `inline=False` | No embeber en el notebook |

In [ ]:
# Ejemplo: lanzar con enlace público y autenticación
# (descomenta para probar)

# demo.launch(
#     share=True,
#     auth=("admin", "1234"),
#     debug=True
# )

### 7.3 Despliegue permanente en Hugging Face Spaces

**Hugging Face Spaces** es la forma recomendada para alojar una app de Gradio de forma **gratuita y permanente**.

**Pasos:**
1. Crea una cuenta en https://huggingface.co
2. Ve a https://huggingface.co/new-space
3. Elige un nombre, selecciona el SDK **Gradio** y la visibilidad (público/privado).
4. Sube estos archivos al Space:
   - `app.py` → tu código de Gradio (debe terminar con `demo.launch()`).
   - `requirements.txt` → las dependencias (ej. `gradio`, `transformers`, `torch`).
5. El Space se construye automáticamente y queda disponible en una URL pública permanente.

**Ejemplo de `app.py`:**
```python
import gradio as gr
from transformers import pipeline

clasificador = pipeline("sentiment-analysis")

def analizar(texto):
    r = clasificador(texto)[0]
    return {r["label"]: float(r["score"])}

demo = gr.Interface(fn=analizar, inputs="text", outputs="label")
demo.launch()
```

**Ejemplo de `requirements.txt`:**
```
gradio
transformers
torch
```

Documentación detallada: https://huggingface.co/docs/hub/spaces-sdks-gradio

### 7.4 Otras opciones de despliegue

- **Docker:** Empaqueta tu app en un contenedor para desplegar en cualquier nube (AWS, GCP, Azure). Usa `server_name="0.0.0.0"`.
- **Integración en FastAPI:** Monta una app de Gradio dentro de un servidor FastAPI con `gr.mount_gradio_app(app, demo, path="/gradio")`.
- **Servidores propios:** Ejecuta `python app.py` detrás de un *reverse proxy* (nginx) para producción.

In [ ]:
# Ejemplo: montar Gradio dentro de FastAPI
# (requiere: pip install fastapi uvicorn)

# from fastapi import FastAPI
# import gradio as gr
#
# app = FastAPI()
#
# def saludar(nombre):
#     return f"Hola {nombre}"
#
# demo = gr.Interface(fn=saludar, inputs="text", outputs="text")
# app = gr.mount_gradio_app(app, demo, path="/gradio")
#
# # Ejecutar con: uvicorn app:app --host 0.0.0.0 --port 8000
# # La interfaz estará en http://localhost:8000/gradio

## 8. Cerrar los servidores

Cada `launch()` abre un servidor. Para liberarlos todos, ejecuta:

In [ ]:
import gradio as gr
gr.close_all()
print("Todos los servidores de Gradio se han cerrado.")

## 9. Resumen

- **Gradio** convierte funciones de Python en interfaces web interactivas con muy poco código.
- Usa `gr.Interface` para prototipos rápidos y `gr.Blocks` para diseños personalizados.
- `gr.ChatInterface` facilita crear chatbots para LLMs.
- Para **compartir rápido**: `launch(share=True)` (enlace temporal de 72 h).
- Para **producción gratuita y permanente**: **Hugging Face Spaces**.
- Para **entornos propios**: Docker, FastAPI o un servidor con reverse proxy.

**Documentación oficial:** https://www.gradio.app/docs